In [274]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from PIL import Image
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision import models
from torchvision.models import resnet50,ResNet50_Weights
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [275]:
data_dir = Path("images")
dataset = []

for class_folder in data_dir.iterdir():
    if class_folder.name != 'images' and class_folder.is_dir():
        if class_folder.is_dir():
            label = class_folder.name

            for image_file in class_folder.iterdir():
                if image_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                    dataset.append({
                        'image_path': str(image_file),
                        'file_name' : image_file.name,
                        'label': label
                    })

dataset = pd.DataFrame(dataset)
dataset.head()

,image_path,file_name,label
0,images\armature\armature-coil001.jpg,armature-coil001.jpg,armature
1,images\armature\armature-coil002.jpg,armature-coil002.jpg,armature
2,images\armature\armature-coil003.jpg,armature-coil003.jpg,armature
3,images\armature\armature-coil004.jpg,armature-coil004.jpg,armature
4,images\armature\armature-coil005.jpg,armature-coil005.jpg,armature


In [276]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
dataset['label'] = le.fit_transform(dataset['label'])

In [277]:
print(len(set(dataset['label'])))

36


In [278]:
from sklearn.model_selection import train_test_split
datatrain, datatest = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset['label'])
dataval, datatest = train_test_split(datatest, test_size=0.5, random_state=42, stratify=datatest['label'])

In [279]:
img_size = 260

In [280]:
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [281]:
class TrainDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path', label_col='label'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
    self.labels = df[label_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img, torch.tensor(self.labels[index], dtype=torch.long)

class TestDataset(Dataset):
  def __init__(self, df, transform=None, path_col='image_path'):
    self.df = df
    self.transform = transform
    self.path_col = df[path_col].values
  
  def __len__(self):
    return len(self.df)

  def __getitem__(self, index):
    path = str(self.path_col[index])
    img = Image.open(path).convert('RGB')
    if self.transform:
      img = self.transform(img)
    return img

In [282]:
train_dataset = TrainDataset(
    df=datatrain,
    transform=train_transform,
    label_col='label',
    path_col='image_path'
)

validation_dataset = TrainDataset(
    df=dataval,
    transform=test_transform,
    label_col='label',
    path_col='image_path'
)

test_dataset = TestDataset(
    df=datatest,
    transform=test_transform,
    path_col='image_path'
)

In [283]:
BATCH_SIZE=32
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=0
)
validation_loader=DataLoader(
    validation_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=0
)

In [284]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [285]:
class CNN_preantrenat(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)

        in_features = self.model.classifier[1].in_features
        
        self.model.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model=CNN_preantrenat(num_classes=36).to(device)

criterion=nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=0.00005, 
    weight_decay=1e-3
)

scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='min',
    factor=0.5,
    patience=2, 
    verbose=True
)

c:\Users\Andrei\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [286]:
epochs=20
train_losses =[]
val_losses=[]
val_accuracies=[]

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    model.eval()
    running_val_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()
  
            _, predicted = torch.max(outputs.data, 1) 

            
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()
      
    avg_val_loss = running_val_loss / len(validation_loader)
    val_losses.append(avg_val_loss)
    
    avg_val_accuracy = correct_predictions / total_samples
    val_accuracies.append(avg_val_accuracy)

    scheduler.step(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} Validation Loss:{avg_val_loss:.4f} Validation Accuracy: {avg_val_accuracy:.4f} Learning Rate: {current_lr:.6f}")

Epoch [1/20] Train Loss: 3.3406 Validation Loss:2.9064 Validation Accuracy: 0.3157 Learning Rate: 0.000050
Epoch [2/20] Train Loss: 2.5761 Validation Loss:2.3081 Validation Accuracy: 0.4122 Learning Rate: 0.000050
Epoch [3/20] Train Loss: 2.2097 Validation Loss:2.0862 Validation Accuracy: 0.4786 Learning Rate: 0.000050
Epoch [4/20] Train Loss: 1.9993 Validation Loss:1.9805 Validation Accuracy: 0.5068 Learning Rate: 0.000050
Epoch [5/20] Train Loss: 1.8783 Validation Loss:1.9252 Validation Accuracy: 0.5250 Learning Rate: 0.000050
Epoch [6/20] Train Loss: 1.7564 Validation Loss:1.8889 Validation Accuracy: 0.5487 Learning Rate: 0.000050
Epoch [7/20] Train Loss: 1.6603 Validation Loss:1.8692 Validation Accuracy: 0.5523 Learning Rate: 0.000050
Epoch [8/20] Train Loss: 1.5717 Validation Loss:1.8579 Validation Accuracy: 0.5578 Learning Rate: 0.000050
Epoch [9/20] Train Loss: 1.4988 Validation Loss:1.8648 Validation Accuracy: 0.5532 Learning Rate: 0.000050
Epoch [10/20] Train Loss: 1.4329 Vali

In [287]:
model.eval()
test_predictions=[]
test_image_names=[]
test_loader=DataLoader(test_dataset, 
                       batch_size=BATCH_SIZE, 
                       shuffle=False)

In [288]:
with torch.no_grad():
    for images in test_loader:
        images=images.to(device)
        outputs=model(images)
        _,predicted=torch.max(outputs,1)
        test_predictions.extend(predicted.cpu().numpy())


In [289]:
datatest

,image_path,file_name,label
8474,images\relay\relay265.jpg,relay265.jpg,26
8179,images\pulse-generator\pulse-generator257.jpg,pulse-generator257.jpg,25
8013,images\pulse-generator\pulse-generator091.jpg,pulse-generator091.jpg,25
1942,images\Electrolytic-capacitor\Electrolytic-cap...,Electrolytic-capacitor1166.jpg,1
5135,images\limiter-clipper\limiter-clipper188.jpg,limiter-clipper188.jpg,16
...,...,...,...
6927,images\PNP-transistor\PNP-transistor017.jpg,PNP-transistor017.jpg,4
8016,images\pulse-generator\pulse-generator094.jpg,pulse-generator094.jpg,25
8354,images\relay\relay115.jpg,relay115.jpg,26
5059,images\limiter-clipper\limiter-clipper112.jpg,limiter-clipper112.jpg,16


In [290]:
test_predictions = le.inverse_transform(test_predictions)

In [291]:
len(test_predictions)

1099

In [292]:
datatest = datatest.reset_index(drop=True)
output = []

for index, row in datatest.iterrows():
  output.append({
    'datapointID' : row['file_name'],
    'answer' : test_predictions[index]
  })
output = pd.DataFrame(output)
output

,datapointID,answer
0,relay265.jpg,step-down-transformer
1,pulse-generator257.jpg,pulse-generator
2,pulse-generator091.jpg,pulse-generator
3,Electrolytic-capacitor1166.jpg,Electrolytic-capacitor
4,limiter-clipper188.jpg,limiter-clipper
...,...,...
1094,PNP-transistor017.jpg,junction-transistor
1095,pulse-generator094.jpg,limiter-clipper
1096,relay115.jpg,electric-relay
1097,limiter-clipper112.jpg,Bypass-capacitor


In [293]:
output.to_csv('submission.csv', index = False)